# Reading and importing libraries for data

In [132]:
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [133]:
df = pd.read_csv('./Data/train.csv')

In [134]:
df.columns

Index(['CustomerID', 'State', 'Customer Lifetime Value', 'Response',
       'Coverage', 'Coverage Index', 'Education', 'Education Index',
       'Effective To Date', 'Employment Status', 'Employment Status Index',
       'Gender', 'Income', 'Marital Status', 'Marital Status Index',
       'Months Since Last Claim', 'Months Since Policy Inception',
       'Number of Open Complaints', 'Number of Policies', 'Policy Type',
       'Policy Type Index', 'Policy', 'Policy Index', 'Renew Offer Type',
       'Sales Channel', 'Sales Channel Index', 'Vehicle Size',
       'Vehicle Size Index', 'Claim over 1k'],
      dtype='object')

# Data Exploration

In [135]:
columns = df.columns

In [136]:
# Create the DataFrame with Unique_Counts
nunique_df = pd.DataFrame({
    'Column': columns,
    'Unique_Counts': [df[col].nunique() for col in columns],
    'Data_Type': [df[col].dtype for col in columns]  # Adding data types
})

# Add the Description column based on Unique_Counts
nunique_df['Description'] = [
    df[col].unique().tolist() if df[col].nunique() < 10 else None for col in columns
]

In [137]:
nunique_df

,Column,Unique_Counts,Data_Type,Description
0,CustomerID,7290,object,None
1,State,5,object,"[California, Washington, Oregon, Arizona, Nevada]"
2,Customer Lifetime Value,6464,float64,None
3,Response,2,object,"[No, Yes]"
4,Coverage,3,object,"[Basic, Extended, Premium]"
5,Coverage Index,3,int64,"[0, 1, 2]"
6,Education,5,object,"[Bachelor, High School or Below, College, Mast..."
7,Education Index,5,int64,"[2, 0, 1, 3, 4]"
8,Effective To Date,59,object,None
9,Employment Status,5,object,"[Employed, Unemployed, Retired, Medical Leave,..."


# Data Cleaning for Categorical Var

## Creating Mapping Dictionary

In [138]:
# New mapping dict for these categorical variables
mapping_dict = {
    'State': {
        'California': 0,
        'Washington': 1,
        'Oregon': 2,
        'Arizona': 3,
        'Nevada': 4
    },
    'Response': {
        'No': 0,
        'Yes': 1
    },
    'Gender': {
        'F': 0,
        'M': 1
    },
    'Sales Channel': {
        'Web': 0,
        'Branch': 1,
        'Agent': 2,
        'Call Center': 3
    }
}

# already existing mapping
mapping_dict_existing = {
    'Coverage': {
        'Basic': 0,
        'Extended': 1,
        'Premium': 2
    },
    'Education': {
        'High School': 0,
        'Bachelor': 1,
        'Master': 2,
        'PhD': 3
    },
    'Employment Status': {
        'Unemployed': 0,
        'Employed': 1,
        'Medical Leave': 2,
        'Retired': 3
    },
    'Marital Status': {
        'Single': 0,
        'Married': 1,
        'Divorced': 2
    },
    'Policy Type': {
        'Personal Auto': 0,
        'Corporate Auto': 1,
        'Special Auto': 2
    },
    'Policy': {
        'Personal L1': 0,
        'Personal L2': 1,
        'Personal L3': 2,
        'Corporate L1': 3,
        'Corporate L2': 4,
        'Corporate L3': 5,
        'Special L1': 6,
        'Special L2': 7,
        'Special L3': 8
    },
    'Vehicle Size': {
        'Small': 0,
        'Medium': 1,
        'Large': 2
    }
}

In [139]:
df_new=df.copy()

In [140]:
for col, mapping in mapping_dict.items():
    if col in df_new.columns:
        # Apply the mapping from the dictionary to the corresponding column in df_new
        df_new[col] = df_new[col].map(mapping)

In [141]:
# drop necessary columns
df_new = df_new.drop(columns=['Coverage', 'Education', 'Employment Status', 'Marital Status', 
                      'Policy Type', 'Policy', 'Vehicle Size'])

## Data cleaning for Dates

In [142]:
# We will make a nother column called days to calculate the date.
# This 'days' column will contain the number of days since the min date in the data

# Ensure the date column is in datetime format
df_new['Effective To Date'] = pd.to_datetime(df_new['Effective To Date'])

# Find the minimum date
min_date = df_new['Effective To Date'].min()

# Create a new column 'Days' that represents the number of days since the minimum date
df_new['Days'] = (df_new['Effective To Date'] - min_date).dt.days.astype(np.int64)


In [143]:
# we will drop the date for now
df_new.drop(columns=['Effective To Date'], inplace = True)

In [144]:
#df_new['Effective To Date'] = pd.to_datetime(df_new['Effective To Date'])

In [145]:
# Extract useful features from the date column
# Might be better removing the dates because each Year, Month, and Date are being processed as separate categories 
#   This loses the purpose of date.
#df_new['Effective_Year'] = df_new['Effective To Date'].dt.year
#df_new['Effective_Month'] = df_new['Effective To Date'].dt.month
#df_new['Effective_Day'] = df_new['Effective To Date'].dt.day

# Mutual Information Feature Selection

In [146]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7290 entries, 0 to 7289
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   CustomerID                     7290 non-null   object 
 1   State                          7290 non-null   int64  
 2   Customer Lifetime Value        7290 non-null   float64
 3   Response                       7290 non-null   int64  
 4   Coverage Index                 7290 non-null   int64  
 5   Education Index                7290 non-null   int64  
 6   Employment Status Index        7290 non-null   int64  
 7   Gender                         7290 non-null   int64  
 8   Income                         7290 non-null   int64  
 9   Marital Status Index           7290 non-null   int64  
 10  Months Since Last Claim        7290 non-null   int64  
 11  Months Since Policy Inception  7290 non-null   int64  
 12  Number of Open Complaints      7290 non-null   i

In [147]:
# Define the features (X) and the target variable (y)
X = df_new.drop(columns=['Claim over 1k', 'CustomerID'])  # Exclude target and ID
y = df_new['Claim over 1k']  # Target variable

# Step 3: Calculate Mutual Information for Classification
mi_scores = mutual_info_classif(X, y)

# Create a DataFrame to show MI scores for each feature
mi_df = pd.DataFrame({'Feature': X.columns, 'Mutual Information': mi_scores})

# Sort the features by MI score in descending order
mi_df = mi_df.sort_values(by='Mutual Information', ascending=False)

In [148]:
# Get the list of features with MI score greater than 0
selected_features = mi_df[mi_df['Mutual Information'] > 0]['Feature'].to_list()

# Append the 'Claim over 1k' and 'CustomerID' to the selected features
final_features = selected_features + ['Claim over 1k', 'CustomerID']

In [149]:
final_df = df_new[final_features]

## Model selection

This is a classification problem since we have feautres selected and we want to classify, given the features, weather or not claim over 1k is true or false.

I am planning to use k fold cross validation to test multiple models and find the best averaged scored model.

In [150]:
# Assuming 'df_new' is your DataFrame and 'Claim over 1k' is the target variable
X = df_new.drop(columns=['Claim over 1k', 'CustomerID'])  # Features
y = df_new['Claim over 1k']  # Target

# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [151]:
# Define models you want to test
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC()
}

In [152]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=5)  # 5-fold cross-validation

In [153]:
# Automated model evaluation storage
results_list = []  # Use a list to store intermediate results

# Loop through models
for model_name, model in models.items():
    # Perform cross-validation
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')  # Use 'accuracy' as the evaluation metric
    avg_score = scores.mean()
    
    # Store the results in a dictionary first
    results_list.append({'Model': model_name, 'Average Score': avg_score})

# Convert the list of results into a DataFrame
results_df = pd.DataFrame(results_list)

c:\Users\06pau\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\06pau\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\06pau\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\06pau\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\06pau\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the 

In [154]:
# Display the results
results_df

,Model,Average Score
0,Logistic Regression,0.879698
1,Random Forest,0.907682
2,SVM,0.886694


: 